# A/A Test: V3 Customer Lookalike

Validates the V3 Customer Lookalike synthetic control by running it against
historical non-campaign periods. Expected result: **zero uplift**.

| Verdict | Condition |
|---------|----------|
| PASS | |mean uplift| <= 2% AND 95% CI contains zero |
| WARNING | |mean uplift| > 2% OR 95% CI excludes zero |
| HARD FAIL | |mean uplift| > 5% |

---
## 1. Config & Authentication

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
import time, gc, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

# Mount Google Drive (one-time auth prompt)
from google.colab import drive
drive.mount('/content/drive')

# Point Python to the config directory on Drive
# Upload aa_config.py + campaign_data.csv to this folder ONCE
import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)  # so aa_config can find campaign_data.csv

import aa_config as cfg

client = bigquery.Client(project=cfg.PROJECT_ID)
print(f'Connected to {cfg.PROJECT_ID}')
print(f'Markets: {cfg.COUNTRIES} | Seeds: {cfg.N_SEEDS}')
print(f'Windows: {sum(len(cfg.TIME_WINDOWS.get(c,[])) for c in cfg.COUNTRIES)}')

---
## 2. Campaign Overlap Validation

In [ ]:
def validate_no_campaign_overlap(client, countries, time_windows):
    checks = []
    for country in countries:
        for w in time_windows.get(country, []):
            label = f"{w['start']} to {w['post_end']}"
            for table in [
                'just-data-warehouse.customer_intelligence.customer_lookalike_evaluation_audience',
                'just-data-warehouse.customer_intelligence.customer_lookalike_evaluation_audience_citytype',
            ]:
                query = f"""
                SELECT COUNT(*) AS cnt FROM `{table}`
                WHERE country = '{country}'
                  AND (campaign_start_date BETWEEN '{w['start']}' AND '{w['post_end']}'
                    OR campaign_end_date BETWEEN '{w['start']}' AND '{w['post_end']}')
                """
                try:
                    cnt = client.query(query).to_dataframe()['cnt'].iloc[0]
                except Exception:
                    cnt = -1
                checks.append({'country': country, 'window': label,
                               'source': table.split('.')[-1], 'overlapping_rows': cnt})
    df = pd.DataFrame(checks)
    overlaps = df[df['overlapping_rows'] > 0]
    if len(overlaps) > 0:
        print('WARNING: Overlap detected:')
        print(overlaps.to_string(index=False))
    else:
        print('PASS: No campaign overlap detected.')
    return df

df_overlap = validate_no_campaign_overlap(client, cfg.COUNTRIES, cfg.TIME_WINDOWS)
df_overlap

---
## 3. Audience Creation

In [ ]:
def create_aa_audience(client, country, window, seed):
    start = window['start']
    query = f"""
    SELECT fo.customerid, dc.country,
        CASE WHEN dcu.optin = true THEN 1 ELSE 0 END AS optin,
        COALESCE(s.base_segment, 'unknown') AS base_segment
    FROM `just-data-warehouse.dwh.fact_order` AS fo
    INNER JOIN `just-data-warehouse.dwh.dim_customer` AS dcu ON fo.customerid = dcu.customerid
    INNER JOIN `just-data-warehouse.dwh.dim_country` AS dc ON fo.countryid = dc.countryid
    LEFT JOIN `just-data-warehouse.core_dwh.dim_unique_customer_history` AS ch
        ON fo.customerid = ch.customer_id AND ch.snapshot_date = DATE('{start}')
    LEFT JOIN `just-data-warehouse.dwh.fact_segmentation_scv_key` AS s
        ON ch.scv_key = s.scv_key AND s.snapshot_date = DATE_SUB(DATE('{start}'), INTERVAL 1 DAY)
    WHERE dc.country IN ({cfg.country_codes_sql(country)})
      AND DATE(orderdatetime) >= DATE_SUB(DATE('{start}'), INTERVAL 365 DAY)
      AND DATE(orderdatetime) < DATE('{start}')
      AND DATE(orderdatetime) >= DATE('2021-01-01')
    GROUP BY 1, 2, 3, 4
    HAVING SUM(nroforders) > 0
    """
    df = client.query(query).to_dataframe()

    # Sample down if audience is too large (DE has 15M+ active customers)
    if len(df) > cfg.MAX_AUDIENCE_SIZE:
        print(f'  Sampling {cfg.MAX_AUDIENCE_SIZE:,} from {len(df):,} customers')
        df = df.sample(n=cfg.MAX_AUDIENCE_SIZE, random_state=seed).reset_index(drop=True)

    rng = np.random.RandomState(seed)
    df['treatment'] = 0
    for seg in df['base_segment'].unique():
        mask = df['base_segment'] == seg
        df.loc[mask, 'treatment'] = rng.binomial(1, cfg.TREATMENT_SHARE, size=mask.sum())

    campaign_name = f"{cfg.AA_PREFIX_V3}_{country}_{start}_seed{seed}"
    df['campaign_name'] = campaign_name
    df['campaign_start_date'] = pd.to_datetime(window['start']).date()
    df['campaign_end_date'] = pd.to_datetime(window['end']).date()
    df['post_period_start_date'] = pd.to_datetime(window['post_start']).date()
    df['post_period_end_date'] = pd.to_datetime(window['post_end']).date()
    n_t = (df['treatment']==1).sum()
    n_c = (df['treatment']==0).sum()
    print(f'  {country}|{start}|seed{seed}: {len(df)} ({n_t} treat, {n_c} ctrl)')
    return df, campaign_name


def upload_aa_audience(client, df, table_id):
    cols = ['customerid','country','campaign_name','treatment',
            'campaign_start_date','campaign_end_date',
            'post_period_start_date','post_period_end_date']
    schema = [bigquery.SchemaField(c, 'STRING' if c in ('customerid','country','campaign_name')
              else 'INTEGER' if c == 'treatment' else 'DATE') for c in cols]
    job = client.load_table_from_dataframe(
        df[cols], table_id,
        job_config=bigquery.LoadJobConfig(schema=schema, write_disposition='WRITE_APPEND'))
    job.result()
    print(f'  Uploaded {len(df)} rows')


def cleanup_aa_audience(client, table_id, campaign_name):
    client.query(f"DELETE FROM `{table_id}` WHERE campaign_name = '{campaign_name}'").result()

print('Audience functions loaded.')

---
## 4. V3 Matching (BQ-native exact match + local KNN)

In [ ]:
def run_matching_v3(client, country, date_str, campaign_name):
    feature_subquery = f"""
    SELECT *,
        NTILE(3) OVER (PARTITION BY country ORDER BY L365D_AOV) AS L365D_AOV_cat
    FROM (
        SELECT fo.customerid, lc.country,
            CASE WHEN dcu.optin = true THEN 1 ELSE 0 END AS optin,
            lc.campaign_start_date, lc.campaign_end_date,
            '{campaign_name}' AS campaign_name, lc.treatment,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -7 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) AS L7D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -14 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) AS L14D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) AS L30D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -90 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) AS L90D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) AND restaurant_discount > 0
                THEN nroforders ELSE 0 END) AS L30D_promo_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -180 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) AS L180D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) AS L365D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -730 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) AS L730D_orders,
            SAFE_DIVIDE(
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY)
                          AND DATE(orderdatetime) < DATE(campaign_start_date) THEN food_total ELSE 0 END),
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY)
                          AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END)
            ) AS L365D_AOV,
            SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date)
                      AND DATE(orderdatetime) <= DATE(campaign_end_date) THEN nroforders ELSE 0 END) AS campaign_period_orders,
            SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date)
                      AND DATE(orderdatetime) <= DATE(campaign_end_date) AND restaurant_discount > 0
                THEN nroforders ELSE 0 END) AS campaign_period_promo_orders,
            SUM(CASE WHEN DATE(orderdatetime) > DATE(post_period_start_date)
                      AND DATE(orderdatetime) <= DATE(post_period_end_date) THEN nroforders ELSE 0 END) AS post_period_orders
        FROM `just-data-warehouse.dwh.fact_order` AS fo
        INNER JOIN `just-data-warehouse.dwh.dim_customer` AS dcu ON fo.customerid = dcu.customerid
        INNER JOIN `just-data-warehouse.dwh.dim_country` AS dc ON fo.countryid = dc.countryid
        INNER JOIN `{cfg.AA_AUDIENCE_TABLE}` AS lc ON fo.customerid = lc.customerid
        WHERE DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -730 DAY)
          AND DATE(orderdatetime) >= DATE('2020-01-01')
          AND lc.campaign_name = '{campaign_name}' AND lc.country IN ({cfg.country_codes_sql(country)})
          AND DATE(lc.campaign_start_date) = '{date_str}'
        GROUP BY 1,2,3,4,5,6,7
    )
    """

    outlier = """L7D_orders<=7 AND L14D_orders<=14 AND L30D_orders<=30
        AND L90D_orders<=90 AND L180D_orders<=180 AND L365D_orders<=365
        AND L730D_orders<=730 AND L30D_promo_orders=0"""

    match_keys = [c for c in cfg.LOOK_ALIKE_CONDITIONS_V3
                  if c not in ('country','campaign_start_date')]
    match_join = ' AND '.join([f't.{c}=l.{c}' for c in match_keys])

    # --- Exact match in BQ ---
    exact_q = f"""
    CREATE TEMP TABLE features AS {feature_subquery};
    CREATE TEMP TABLE treatment AS SELECT * FROM features WHERE treatment=1 AND {outlier};
    CREATE TEMP TABLE lookalike AS SELECT * FROM features WHERE treatment=0 AND {outlier};
    CREATE TEMP TABLE exact_matches AS
    SELECT t.customerid AS customerid_treatment, t.country, t.optin, t.campaign_start_date,
        t.L30D_orders AS L30D_orders_treatment,
        t.campaign_period_orders AS campaign_period_orders_treatment,
        t.campaign_period_promo_orders AS campaign_period_promo_orders_treatment,
        t.post_period_orders AS post_period_orders_treatment,
        AVG(l.L30D_orders) AS L30D_orders_lookalike,
        AVG(l.campaign_period_orders) AS campaign_period_orders_lookalike,
        AVG(l.campaign_period_promo_orders) AS campaign_period_promo_orders_lookalike,
        AVG(l.post_period_orders) AS post_period_orders_lookalike,
        'exact_match' AS source
    FROM treatment t
    INNER JOIN lookalike l ON t.country=l.country AND t.campaign_start_date=l.campaign_start_date AND {match_join}
    GROUP BY 1,2,3,4,5,6,7,8;

    SELECT *, 'exact' AS result_type FROM exact_matches
    UNION ALL
    SELECT t.customerid, t.country, t.optin, t.campaign_start_date,
        t.L30D_orders, t.campaign_period_orders, t.campaign_period_promo_orders,
        t.post_period_orders,
        CAST(NULL AS FLOAT64), CAST(NULL AS FLOAT64), CAST(NULL AS FLOAT64), CAST(NULL AS FLOAT64),
        'unmatched', 'unmatched'
    FROM treatment t WHERE t.customerid NOT IN (SELECT customerid_treatment FROM exact_matches);
    """
    df = client.query(exact_q).to_dataframe()
    if len(df) == 0:
        return None
    df_exact = df[df['result_type']=='exact'].drop(columns=['result_type'])
    df_unmatched = df[df['result_type']=='unmatched'].drop(columns=['result_type'])
    del df; gc.collect()
    print(f'  Exact: {len(df_exact)} | Unmatched: {len(df_unmatched)}')

    # --- KNN for unmatched (standalone BQ query + local sklearn) ---
    nn_result = pd.DataFrame()
    if len(df_unmatched) > 0:
        la_q = f"""WITH features AS ({feature_subquery})
        SELECT customerid,optin,L14D_orders,L30D_orders,L90D_orders,
               L180D_orders,L365D_orders,L730D_orders,
               campaign_period_orders,campaign_period_promo_orders,post_period_orders
        FROM features WHERE treatment=0 AND {outlier}"""
        la = client.query(la_q).to_dataframe()

        ids_str = ','.join([f"'{x}'" for x in df_unmatched['customerid_treatment']])
        un_q = f"""WITH features AS ({feature_subquery})
        SELECT customerid,optin,L14D_orders,L30D_orders,L90D_orders,
               L180D_orders,L365D_orders,L730D_orders,
               campaign_period_orders,campaign_period_promo_orders,post_period_orders
        FROM features WHERE treatment=1 AND customerid IN ({ids_str}) AND {outlier}"""
        un = client.query(un_q).to_dataframe()

        if len(la) > 0 and len(un) > 0:
            sc = ['optin','L14D_orders','L30D_orders','L90D_orders','L180D_orders','L365D_orders','L730D_orders']
            la[sc] = la[sc].fillna(0)
            un[sc] = un[sc].fillna(0)
            scaler = MinMaxScaler()
            knn = NearestNeighbors(n_neighbors=1)
            knn.fit(scaler.fit_transform(la[sc]))
            dist, idx = knn.kneighbors(scaler.transform(un[sc]))
            m = la.iloc[idx.flatten()]
            nn_result = pd.DataFrame({
                'customerid_treatment': un['customerid'].values,
                'country': country, 'optin': un['optin'].values,
                'campaign_start_date': pd.to_datetime(date_str).date(),
                'L30D_orders_treatment': un['L30D_orders'].values,
                'campaign_period_orders_treatment': un['campaign_period_orders'].values,
                'campaign_period_promo_orders_treatment': un['campaign_period_promo_orders'].values,
                'post_period_orders_treatment': un['post_period_orders'].values,
                'L30D_orders_lookalike': m['L30D_orders'].values,
                'campaign_period_orders_lookalike': m['campaign_period_orders'].values,
                'campaign_period_promo_orders_lookalike': m['campaign_period_promo_orders'].values,
                'post_period_orders_lookalike': m['post_period_orders'].values,
                'source': 'nearest_neighbor'})
            print(f'  KNN matched: {len(nn_result)}')
        del la, un; gc.collect()

    parts = [x for x in [df_exact, nn_result] if len(x)>0]
    if not parts: return None
    final = pd.concat(parts, ignore_index=True)
    final['campaign_name'] = campaign_name
    del df_exact, nn_result; gc.collect()
    print(f'  Total: {len(final)}')
    return final

print('V3 matching loaded.')

---
## 5. Metrics & Write to BQ

In [ ]:
def safe_divide(a, b):
    return np.where(b > 0, a / b, np.nan)


def calculate_run_metrics(df):
    """Aggregate matched results into per-run uplift metrics."""
    agg = {
        'n_units': ('customerid_treatment', 'nunique'),
        'n_exact': ('source', lambda x: (x=='exact_match').sum()),
        'n_knn': ('source', lambda x: (x=='nearest_neighbor').sum()),
        'L30D_t': ('L30D_orders_treatment', 'sum'),
        'L30D_l': ('L30D_orders_lookalike', 'sum'),
        'camp_t': ('campaign_period_orders_treatment', 'sum'),
        'camp_l': ('campaign_period_orders_lookalike', 'sum'),
        'post_t': ('post_period_orders_treatment', 'sum'),
        'post_l': ('post_period_orders_lookalike', 'sum'),
    }
    # Total level
    t = df.groupby(['country']).agg(**agg).reset_index()
    t['base_segment'] = 'total'
    # Segment level
    s = df.groupby(['country','base_segment']).agg(**agg).reset_index()
    out = pd.concat([t, s], ignore_index=True)
    out['exact_match_pct'] = safe_divide(out['n_exact'], out['n_units'])
    out['pre_period_uplift'] = safe_divide(out['L30D_t'], out['L30D_l']) - 1
    out['campaign_period_uplift'] = safe_divide(out['camp_t'], out['camp_l']) - 1
    out['post_period_uplift'] = safe_divide(out['post_t'], out['post_l']) - 1
    out['campaign_incr_orders'] = out['camp_t'] - out['camp_l']
    out['post_incr_orders'] = out['post_t'] - out['post_l']
    return out


def write_results(client, metrics, window_start, window_end, seed):
    """Write run metrics to the shared AA results table."""
    df = metrics[['country','base_segment','n_units','exact_match_pct',
                  'pre_period_uplift','campaign_period_uplift','post_period_uplift',
                  'campaign_incr_orders','post_incr_orders']].copy()
    df['technique'] = 'V3_CUSTOMER'
    df['window_start'] = window_start
    df['window_end'] = window_end
    df['seed'] = seed
    df['treatment_city'] = None
    df['n_controls'] = (metrics['n_exact'] + metrics['n_knn']).values
    df['run_timestamp'] = pd.Timestamp.now(tz='Europe/Amsterdam')

    schema = [
        bigquery.SchemaField('technique', 'STRING'),
        bigquery.SchemaField('country', 'STRING'),
        bigquery.SchemaField('window_start', 'STRING'),
        bigquery.SchemaField('window_end', 'STRING'),
        bigquery.SchemaField('seed', 'INTEGER'),
        bigquery.SchemaField('treatment_city', 'STRING'),
        bigquery.SchemaField('base_segment', 'STRING'),
        bigquery.SchemaField('n_units', 'INTEGER'),
        bigquery.SchemaField('n_controls', 'INTEGER'),
        bigquery.SchemaField('exact_match_pct', 'FLOAT'),
        bigquery.SchemaField('pre_period_uplift', 'FLOAT'),
        bigquery.SchemaField('campaign_period_uplift', 'FLOAT'),
        bigquery.SchemaField('post_period_uplift', 'FLOAT'),
        bigquery.SchemaField('campaign_incr_orders', 'FLOAT'),
        bigquery.SchemaField('post_incr_orders', 'FLOAT'),
        bigquery.SchemaField('run_timestamp', 'TIMESTAMP'),
    ]
    select_cols = [s.name for s in schema]
    job = client.load_table_from_dataframe(
        df[select_cols], cfg.AA_RESULTS_TABLE,
        job_config=bigquery.LoadJobConfig(schema=schema, write_disposition='WRITE_APPEND'))
    job.result()
    print(f'  Wrote {len(df)} rows to {cfg.AA_RESULTS_TABLE}')

print('Metrics & write functions loaded.')

---
## 6. Run A/A Test

In [ ]:
start_time = time.time()
total = sum(len(cfg.TIME_WINDOWS.get(c,[])) for c in cfg.COUNTRIES) * cfg.N_SEEDS
run = 0

for country in cfg.COUNTRIES:
    for window in cfg.TIME_WINDOWS.get(country, []):
        for seed in range(cfg.N_SEEDS):
            run += 1
            cname = f"{cfg.AA_PREFIX_V3}_{country}_{window['start']}_seed{seed}"
            print(f'\n=== Run {run}/{total}: {cname} ===')
            df_aud = None
            try:
                df_aud, cname = create_aa_audience(client, country, window, seed)
                upload_aa_audience(client, df_aud, cfg.AA_AUDIENCE_TABLE)

                result = run_matching_v3(client, country, window['start'], cname)
                if result is not None and len(result) > 0:
                    seg_map = df_aud[['customerid','base_segment']].drop_duplicates()
                    result = result.merge(seg_map, left_on='customerid_treatment',
                                          right_on='customerid', how='left'
                                  ).drop(columns=['customerid'], errors='ignore')
                    result['base_segment'] = result['base_segment'].fillna('unknown')

                    metrics = calculate_run_metrics(result)
                    write_results(client, metrics, window['start'], window['end'], seed)
                    del result, metrics
                else:
                    print('  WARNING: No results')

            except Exception as e:
                print(f'  ERROR: {e}')
                import traceback; traceback.print_exc()
            finally:
                try: cleanup_aa_audience(client, cfg.AA_AUDIENCE_TABLE, cname)
                except: pass
                if df_aud is not None: del df_aud
                gc.collect()

elapsed = (time.time() - start_time) / 60
print(f'\n{"="*50}')
print(f'V3 A/A test complete in {elapsed:.1f} minutes.')
print(f'Results written to {cfg.AA_RESULTS_TABLE}')

---
## 7. Quick Summary

In [ ]:
df_check = client.query(f"""
    SELECT base_segment, COUNT(*) AS n_rows,
        AVG(campaign_period_uplift) AS mean_uplift,
        AVG(exact_match_pct) AS mean_exact_pct
    FROM `{cfg.AA_RESULTS_TABLE}`
    WHERE technique = 'V3_CUSTOMER'
    GROUP BY 1 ORDER BY 1
""").to_dataframe()
print('Results in BQ:')
df_check